<table>
<tr><td><img style="height: 150px;" src="images/geo_hydro1.jpg"></td>
<td bgcolor="#FFFFFF">
    <p style="font-size: xx-large; font-weight: 900; line-height: 100%">AG Dynamics of the Earth</p>
    <p style="font-size: large; color: rgba(0,0,0,0.5);">Juypter notebooks</p>
    <p style="font-size: large; color: rgba(0,0,0,0.5);">Georg Kaufmann</p>
    </td>
</tr>
</table>

# 05: Natural neighbors explained
----
*Georg Kaufmann,
Geophysics Section,
Institute of Geological Sciences,
Freie Universität Berlin,
Germany*

In [ ]:
import numpy as np
import scipy.optimize
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib import cm
from ipywidgets import interact, interactive, fixed
import ipywidgets as widgets

## Preparation: Create color bars

Create different color maps ...

In [ ]:
# color bars
n_colors = 9
mycolors_bold = ['#FF0000', '#0000FF', '#00FF00']
mycolors_light = ['#FFAAAA','#AAAAFF', '#AAFFAA']
cmap_bold = ListedColormap(mycolors_bold)
cmap_light = ListedColormap(mycolors_light)
cmap_set1 = cm.get_cmap('Set1', 9) # Set1,Set2,Set3

## Preparation: Create random points 

Create random points $X_{ij}, i=1,n; j=1,2$ in the domain $X_{ij} \in [-1,1]$ 
and group them into **two bins**, labeling them
$y_i[0,1]$.

The $X_{ij}$ are our **features**, the $y_i$ our **labels**.

In [ ]:
# create sample data
np.random.seed(112)
n=60
# features
X=np.zeros(2*n).reshape(n,2)
X[:,0] = np.random.uniform(-1,1,n)
X[:,1] = np.random.uniform(-1,1,n)
# labels
y=np.ones(n,dtype=int)
for i in range(n):
    if (X[i,0] <= 0.4):
        y[i] = 0
#print(y)
col_list = [cmap_set1(y[i]) for i in range(n)]

fig,ax = plt.subplots(1, 1,figsize=(6,6))
plt.xlim([-1.1,1.1]);plt.xlabel('x$_1$')
plt.ylim([-1.1,1.1]);plt.ylabel('x$_2$')
ax.scatter(X[:,0],X[:,1],c=col_list,s=100)
plt.show()

----
## Method: k Neighbors

Define **new** point $x^0$ and figure out, to which group of points it belongs,
using a **neighborhood algorithm**:

1. Load data set with $n$ points.
2. Choose **number of neighbors**, $k$.
3. For each point, $i=1,n$:
    - calculate Euclidian distance to other $n-1$ points:
    $$
    dist_i(j) = \sqrt{\left[ x_i-x_j \right]^2}
     $$
    - Sort list of Euclidian distances for point $i$: $sort(dist_i(j))$.
    - Take $k$ first elements of $sort(dist_i(j))$, sum up labels of neighboring points, divide by $k$.
    - Assign new label to point $i$.


In [ ]:
x0 = np.array([0.45,0.52])
r0=0.12
r1=0.22
r2=0.40

fig,ax = plt.subplots(1, 1,figsize=(8,8))
plt.xlim([-1.1,1.1]);plt.xlabel('x$_1$')
plt.ylim([-1.1,1.1]);plt.ylabel('x$_2$')
ax.scatter(X[:,0],X[:,1],c=col_list,s=100)
ax.scatter(x0[0],x0[1],c='g',s=300,edgecolor='black')
theta = np.arange(0, 2.*np.pi, 0.01)
plt.plot(x0[0]+r0 * np.cos(theta), x0[1]+r0 * np.sin(theta),linestyle='-',color='black',label='k=1')
plt.plot(x0[0]+r1 * np.cos(theta), x0[1]+r1 * np.sin(theta),linestyle='--',color='black',label='k=2')
plt.plot(x0[0]+r2 * np.cos(theta), x0[1]+r2 * np.sin(theta),linestyle=':',color='black',label='k=3')
plt.legend()

The three circles arround the new point $x^0$ mark regions with different neighboring points in:
- solid: $k=1$ neighbor (1xblue)
- dashed: $k=2$ neighbors (1xblue, 1xred)
- dotted: $k=3$ neighbors (1xblue, 2xred)

Based on the **labels** of the neighbors, the new point $x^0$ will get his label.

In [ ]:
def knn_predict(x0,X,y,k=3):
    """
    predictor for k^th natural neighbor classifier
    """
    dist = np.zeros(len(y))
    for i in range(len(y)):
        dist[i] = np.sqrt((x0[0]-X[i,0])**2 + (x0[1]-X[i,1])**2)
    nearest = y[np.argsort(dist)]
    #print(nearest)
    label = int(np.round(sum(nearest[0:k])/k,0))
    # plot
    fig,ax = plt.subplots(1, 1,figsize=(8,8))
    plt.xlim([-1.1,1.1]);plt.xlabel('x$_1$')
    plt.ylim([-1.1,1.1]);plt.ylabel('x$_2$')
    ax.scatter(X[:,0],X[:,1],c=col_list,s=100)
    ax.scatter(x0[0],x0[1],edgecolors= "black",c=cmap_set1.colors[label],s=300)
    theta = np.arange(0, 2.*np.pi, 0.01)
    r0 = dist[np.argsort(dist)][k-1]
    plt.plot(x0[0]+r0 * np.cos(theta), x0[1]+r0 * np.sin(theta),linestyle=':',color='black')
    return label

label = knn_predict(x0,X,y,k=1)
print('label: ',label)

In [ ]:
interact(knn_predict, x0=fixed(x0), X=fixed(X), y=fixed(y), k=(1,8,1));

... done